# Choosing a perception radius, and the termination threshold

The zero-randomness run is the **reference**: it disperses to a plateau by pure separation. That
plateau area becomes the stop condition for the randomised runs, so every clip ends at the same
degree of dispersion and randomness changes the *manner* of the motion rather than how far the
swarm got.

Two things have to be true of the radius you pick. The swarm must **settle** rather than expand
until the walls stop it, and it must get there without **too much of the swarm on the walls**.

**Kernel:** `Python (PythonAnalysis)`.

In [ ]:
import sys
from pathlib import Path

ANALYSIS = Path.cwd().parent
if str(ANALYSIS) not in sys.path:
    sys.path.insert(0, str(ANALYSIS))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

import plot_grid as pg
import steady_state as ss

RECORDINGS = ANALYSIS.parent / "Assets/SimulationRecordings/Combinations/Dispersion"

WALL_BUDGET = 0.20        # fraction of agents allowed to touch a wall by the plateau
ARENA_AREA  = ss.DEFAULT_ARENA_AREA

runs = pg.load_runs(RECORDINGS)
print(f"{len(runs)} runs, arena {ARENA_AREA:.0f} u², wall budget {WALL_BUDGET:.0%}")

## 1. The candidates

Only the zero-randomness runs can serve as the reference. `settled` says the hull stopped changing
rather than being cut off by the timeout; `arena_fraction` near 1 means the walls stopped it, not
the separation rule.

In [ ]:
rows = ss.choose_perception(runs, arena_area=ARENA_AREA, budget=WALL_BUDGET)
candidates = pd.DataFrame(rows)
candidates

## 2. Where the walls start to dominate

In [ ]:
fig, (ax_a, ax_w) = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")

by_r = candidates.groupby("perception").agg(
    plateau=("plateau_area", "mean"),
    wall=("wall_fraction", "mean"),
    arena=("arena_fraction", "mean")).reset_index()

ax_a.plot(by_r["perception"], by_r["plateau"], "o-", color="#1f77b4")
ax_a.axhline(ARENA_AREA, color="#d62728", linestyle="--", linewidth=1, label="arena area")
ax_a.set_xscale("log"); ax_a.set_xlabel("perception radius"); ax_a.set_ylabel("plateau area (u²)")
ax_a.legend(); ax_a.grid(alpha=0.25)

ax_w.plot(by_r["perception"], by_r["wall"], "o-", color="#d95f02")
ax_w.axhline(WALL_BUDGET, color="#d62728", linestyle="--", linewidth=1,
             label=f"budget {WALL_BUDGET:.0%}")
ax_w.set_xscale("log"); ax_w.set_xlabel("perception radius")
ax_w.set_ylabel("agents that touched a wall, by plateau")
ax_w.legend(); ax_w.grid(alpha=0.25)

plt.show()

## 3. The pick

The largest radius that settles and stays inside the budget. Its plateau area is the threshold.

In [ ]:
pick = ss.recommend(rows)

if pick is None:
    print("Nothing both settles and stays within budget.")
    print("Widen WALL_BUDGET, lengthen the clips, or sweep smaller radii.")
else:
    TARGET_AREA = pick["plateau_area"]
    print(f"perception {pick['perception']:g}")
    print(f"   plateau      {TARGET_AREA} u²  at {pick['plateau_time']}s")
    print(f"   arena        {pick['arena_fraction']:.0%}")
    print(f"   wall contact {pick['wall_unique']} agents ({pick['wall_fraction']:.0%})")
    print()
    print(f"   -> AbsoluteHullAreaReached target = {TARGET_AREA} u²")

## 4. The reference run, with the threshold drawn on

Sanity check by eye: the line should sit on the flat part, not on the way up.

In [ ]:
ref = [r for r in ss.reference_runs(runs)
       if abs(r["perceptionRadius"] - pick["perception"]) < 1e-9]

fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained")
for r in ref:
    ax.plot(r["times"], r["areas"], linewidth=1.6, label=f"maxSpeed {r['maxSpeed']:g}")

ax.axhline(TARGET_AREA, color="#d62728", linestyle="--", linewidth=1.4,
           label=f"target {TARGET_AREA:g} u²")
ax.axvline(pick["plateau_time"], color="#888888", linestyle=":", linewidth=1,
           label=f"plateau {pick['plateau_time']:g}s")
ax.set_xlabel("time (s)"); ax.set_ylabel("hull area (u²)")
ax.set_title(f"reference, perception {pick['perception']:g}, no randomness")
ax.grid(alpha=0.25); ax.legend()
plt.show()

## 5. Would the threshold work on the randomised runs?

Applies the same rule Unity will, dwell included. Watch two things: does it fire at all before the
timeout, and how far apart the firing times are — that spread becomes a difference in clip length
between conditions.

In [ ]:
DWELL = 0.5

check = pd.DataFrame(ss.verify_threshold(runs, TARGET_AREA, DWELL))
at_pick = check[np.isclose(check["perception"], pick["perception"])]
at_pick

In [ ]:
fired = at_pick[at_pick["fires"]]
if len(fired):
    print(f"fires in {len(fired)} of {len(at_pick)} runs at perception {pick['perception']:g}")
    print(f"   clip length would range {fired['at_time'].min():.1f}s to {fired['at_time'].max():.1f}s")
    print(f"   spread {fired['at_time'].max() - fired['at_time'].min():.1f}s")
    print()
    print("Equalising the endpoint means duration varies instead. Report these alongside results.")
else:
    print("Never fires: the threshold is out of reach for these runs.")

missing = check[~check["fires"]]
if len(missing):
    print(f"\n{len(missing)} run(s) across the whole sweep never reach it:")
    print(missing[["perception", "random", "maxSpeed", "closest_area"]].to_string(index=False))

## 6. Sensitivity

The pick should not hinge on one arbitrary setting. If the recommendation changes wildly with the
budget or the plateau tolerance, say so in the write-up rather than presenting one number.

In [ ]:
for budget in (0.05, 0.10, 0.20, 0.30):
    p = ss.recommend(ss.choose_perception(runs, ARENA_AREA, budget))
    print(f"budget {budget:>4.0%}  ->  " +
          (f"perception {p['perception']:g}, target {p['plateau_area']} u²" if p else "no candidate"))

print()
for slope in (0.01, 0.02, 0.05):
    p = ss.recommend(ss.choose_perception(runs, ARENA_AREA, WALL_BUDGET, slope_tolerance=slope))
    print(f"slope {slope:>5.2f}  ->  " +
          (f"perception {p['perception']:g}, target {p['plateau_area']} u², "
           f"plateau at {p['plateau_time']}s" if p else "no candidate"))

## 7. What to set in Unity

For the Dispersion entry in `SimRecorder`'s per-motion-type rules:

- end condition **Absolute Hull Area**
- **Target Hull Area** = the value printed in section 3
- **Dwell Before Ending** = 0.5 s
- a `Max Time Override` comfortably above the slowest firing time in section 5, so a slow run still
  ends on the rule rather than the timeout